# MathScholar · Step 18b — OCR Repair Re-run on Kaggle GPU

**Project:** MathScholar (team 1) — agentic RAG over *Abramowitz & Stegun,
Handbook of Mathematical Functions*, NBS AMS-55 (1964).
**Step owner:** S1 · Anuron Maitro (2105037) · branch `Anuron-step-18B`
**Plan reference:** `plan.md` Step 18b (depends on Step 18) and §11 (Kaggle walkthrough).

---

## What this notebook is for

Step 16's baseline run (pretrained `facebook/nougat-base`, not fine-tuned) looked
survivable at a glance — 594 of ~1040 pages produced *some* transcript — but it
wasn't: precision on the one gold page we could score was 81%, recall was 28%.
The reader wasn't misreading, it was **stopping early**, and book-wide we had
captured only ~15% of the actual text. Three concrete bugs in `vision/ocr.py`
caused this (full diagnosis in `plan.md` Step 18b): failed pages were discarded
with **no retry** even though `Reader.transcribe_region()` exists for exactly
that; the degeneracy detector's unit-length cap (4 chars) was **shorter than
`\qquad` itself** (6 chars), so the dominant failure mode was invisible to its
own detector; and `generate()` carried **no `repetition_penalty`** at all, which
is what lets a decoder spiral in the first place. This notebook re-runs the
full-book OCR pass with all three fixed, and reports the same four numbers
before/after so the repair is a measured claim, not an assertion:

1. **No-output rate** — plan.md's gate: <=25% to proceed with the 105-page
   training set unchanged; >25% reopens the annotation-budget question.
2. **PDF text-layer coverage** — median per-page and book-wide word coverage
   against the source PDF's own text layer (the diagnostic that found this bug).
3. **char-F1 on the 3 A1 gold pages** — same metric Step 16 reported.
4. **exact-formula-match on the 3 A1 gold pages** — same metric Step 16 reported.

It runs, in order:

| Stage | Module | Owner | What it does here |
|---|---|---|---|
| 1 Ingest | `ingest/loader.py` | S2 | `data/pages/*.png` -> `list[Page]`, `doc_id` = chapter |
| 1 Clean | `ingest/preprocess.py` | S3 | deskew · light denoise · CLAHE (grayscale, no binarisation) |
| 2 Layout | `vision/layout.py` | S1 | two-column split -> blocks -> `list[Region]` in reading order |
| 3 OCR | `vision/ocr.py` | S2 (repaired by S1, Step 18b) | **pretrained** `facebook/nougat-base` -> `.mmd` per page + `list[Chunk]`, now with region-level retry on failure |
| 9 Metrics | `eval/metrics.py` | S3 | char-level F1 + exact-formula-match vs the 3 gold pages |

Note the OCR reader here is still the **pretrained baseline, not fine-tuned**
(`plan.md` Step 11 point 6) — Step 18b fixes inference-path bugs, not model
quality. Fine-tuning is still Sprint 4 / Step 28.

## What it deliberately does NOT do

- **No embedding / no index build.** Steps 14–15 (`embed.py`, `store.py`) sit
  *downstream* of OCR and are not exercised here; the full chain is rebuilt in
  Step 30 once the fine-tuned reader exists. Running BGE-M3 over the whole
  corpus now would burn GPU hours on an index we are going to throw away.
- **No hook seams.** We call the pipeline stages directly rather than
  `pipeline.build_knowledge_base()`, so the `AFTER_OCR` seam (where
  `governance/pii.py` scrubs text) does not fire. That is correct for this step:
  we are producing raw reader output for later inspection and annotation, and
  PII scrubbing belongs to the indexing path that Step 30 runs.
- **No training-set enlargement.** plan.md Step 18b's own analysis (5 numbered
  reasons) is that the 105-page train split stays as-is; this run exists to
  find out whether the repair alone is enough, not to relitigate that decision.

## Deliverables this notebook produces (all inside one downloadable zip)

1. `data/ocr/*.mmd` — one raw Nougat transcript per page (~1040 files)
2. `data/ocr/meta.jsonl` — per-chunk `ocr_confidence` + `bbox` sidecar
3. `data/ocr/failures.json` — **the honest degenerate-page list** (form §5)
4. `step18b_reports/baseline_metrics.{json,md}` — the BEFORE/AFTER numbers
5. `step18b_reports/ocr_coverage.csv` — per-page character/word coverage
6. `step18b_reports/run_manifest.json` — full reproducibility record

## ⚠️ Read this before you run: the two-phase protocol

This notebook has a single switch, `RUN_MODE`, and you are expected to run it
**twice** — smoke first, then full. This is unchanged from Step 16; what
changed is `vision/ocr.py` itself (Step 18b) and `FRESH_START` above, which
makes sure this run cannot silently reuse the pre-repair `.mmd` files.

```
RUN_MODE = "smoke"   ->  20 hand-picked pages, ~30 min   <- DO THIS FIRST
RUN_MODE = "full"    ->  all ~1040 pages,      ~3-4 h
```

Measured stage costs from Step 16 run #1 (2x Tesla T4, 15.6 GB each) — the
repair changes what the OCR stage catches and retries, not these fixed costs:

| stage | smoke (20 pages) | full (~1040 pages) |
|---|---|---|
| clone + deps | ~2 min | ~2 min |
| corpus render (always full book) | **20.1 min** | **20.1 min** |
| loader | 57.7 s | 57.7 s |
| preprocess | 0.1 min | ~5 min |
| layout (incl. TATR load) | 0.6 min | ~30 min |
| **OCR** | ~2 min | **~1.5-3 h** (region-retry on failed pages adds time here) |

**Why the smoke run is not optional, even for a repair re-run.** The region-
retry path (`_retry_page_by_region`, Step 18b defect 1) has never run against
real Nougat inference — it is only verified so far against cached `.mmd` text
and a duck-typed fake reader in `tests/test_ocr.py`. The smoke run is the
first time it touches the actual model.

The asymmetry is stark, and it is the whole argument:

- smoke run that fails  -> you lose ~15 minutes and learn exactly what broke
- full run that fails at page 800, or silently writes 1040 files of garbage
  -> you lose 2–3 hours out of Kaggle's ~30 GPU-hr weekly budget, **and** you
  still have to debug it afterwards

The 20 smoke pages are chosen to span chapters and page *types* — dense numeric
tables (where Nougat's repetition-degeneration failure mode actually bites),
two-column formula pages, and prose — and they **include all three gold pages**
(printed 243, 255, 360), so the smoke run already previews the real BEFORE/AFTER
numbers before you commit to the long job.

**Belt and braces:** even in `"full"` mode this notebook OCRs the smoke subset
*first* and runs an automatic sanity gate on it before touching the rest of the
book. That costs nothing — `ocr.transcribe` skips any page whose `.mmd` already
exists — so the gated pages are simply done early rather than twice.

## Kaggle session settings this notebook expects

Set by `kernel-metadata.json`, so `kaggle kernels push` configures them for you:

| Setting | Value | Why |
|---|---|---|
| Accelerator | **GPU T4 x2** (`NvidiaTeslaT4`) | `configs/config.yaml` sets `device: cuda`; Nougat on CPU is an overnight run |
| Internet | **On** | clones the repo, downloads the A&S PDF and the model weights |
| Persistence | `/kaggle/working` | the only directory that survives into the notebook output |

**Session limits (`plan.md` §11.4).** A session dies at ~9 h interactive / ~12 h
committed, and after ~20 min idle. Push with *Save & Run All* semantics (which
is what `kaggle kernels push` does) so the job survives your laptop closing.
If it does time out, see the **resume** cell below — the OCR loop is resumable
and you restart from where it stopped rather than from zero.

In [ ]:
# ============================================================================
# CONFIGURATION — the only cell you normally edit.
# ============================================================================

# "smoke" = 20 pages (~30 min, includes all 3 gold pages)  <- run this FIRST
# "full"  = every content page (~1040, ~3-4 h)
# Both pay the same fixed ~20 min corpus render; the difference is OCR time.
RUN_MODE = "full"

# Our repo is public, which is what makes this one-line clone possible.
#
# We clone the STEP-18B BRANCH, not main. plan.md Step 18b fixes the bug this run exists
# to repair: failed pages were discarded with no retry, the degeneracy detector could not
# see spirals built from units longer than 4 characters (so `\qquad`, 6 chars, sailed
# through), and generate() carried no repetition_penalty at all. Together these are why
# Step 16 only captured ~15% of the book's words despite 81% precision on what it DID
# transcribe -- the reader wasn't misreading, it was stopping early. Those fixes live on
# this branch until the PR merges, so cloning `main` would silently reproduce the bug.
# After the PR merges, this can go back to "main".
REPO_URL = "https://github.com/anuronmaitro/doc-agent-1.git"
REPO_BRANCH = "Anuron-step-18B"  # must carry the Step 18b vision/ocr.py + tests/test_ocr.py fix

# Step 18b invalidates every .mmd this repo has ever produced -- they were all written by
# the buggy pre-repair reader, so a `.mmd` that already exists on disk is not evidence of
# a good transcript anymore, it is evidence of the bug. FRESH_START=True (the default for
# this repair run) skips the /kaggle/input resume-seed below entirely, so this run OCRs
# every page with the repaired code regardless of what dataset happens to be attached as
# input. Only set it False on a deliberate SAME-run resume, i.e. this Step 18b run itself
# timed out and you reattached ITS OWN interrupted output as the input dataset -- see
# section 11's "if the session timed out mid-run" note.
FRESH_START = True

WORK_DIR = "/kaggle/working"
REPO_DIR = WORK_DIR + "/repo"
STAGE_DIR = WORK_DIR + "/step18b_out"
ZIP_PATH = WORK_DIR + "/mathscholar_step18b_ocr_repair.zip"

# The three A1 gold pages, hand-transcribed in grading_kit/labels.jsonl.
# These are TEST-split pages (summary.md 4h) and give the BEFORE/AFTER number.
GOLD_PRINTED_PAGES = [243, 255, 360]

# 20 smoke pages spanning chapters AND page types. Dense-table pages are in here
# on purpose: that is Nougat's known repetition-degeneration failure mode, so a
# smoke run without tables would be a smoke run that cannot fail informatively.
SMOKE_PRINTED_PAGES = [
    1, 65, 100, 227, 243, 255, 295, 331, 360, 435,
    503, 587, 685, 721, 803, 875, 925, 1011, 1019, 1028,
]

# requirements.lock pins transformers==4.44.2; we install the same version on
# Kaggle so the reader behaves identically to a teammate's local run.
TRANSFORMERS_PIN = "4.44.2"

# Source PDF layout, matching scripts/get_data.sh exactly -- needed by section 8's
# PDF-text-layer coverage measurement (the diagnostic that found this bug in the first
# place: OCR word count vs. the PDF's own text layer, per page, is a ground-truth proxy
# for "how much of the page did we actually capture" that a >=200-char length gate
# cannot see).
PDF_PATH = "data/raw/handbookofmathem1964abra.pdf"
FRONT_MATTER_OFFSET = 32  # printed N = PDF N + 32

# Pre-repair reference numbers, measured on Step 16's output and written up in plan.md
# Step 18b / the diagnostic session in prompt_elias.txt. Hardcoded (not recomputed here)
# because after FRESH_START wipes the old .mmd files there is nothing left on disk to
# recompute them from -- this is the same pattern as FP32_BASELINE_CHAR_F1 below, a fixed
# reference an in-progress run reports its delta against. NOT a claim about this run;
# purely the "before" column.
BEFORE_REPAIR = {
    "naive_failure_rate": 0.429,  # 446/1040 pages hit continue() and were discarded
    "book_wide_word_coverage_vs_pdf_text_layer": 0.151,  # 15.1% of the book's words captured
    "median_page_coverage_vs_pdf_text_layer": 0.28,  # median among "successful" pages
    "as_p0360_precision": 0.813,  # what it transcribed was accurate...
    "as_p0360_recall": 0.280,  # ...but it covered only 28% of the page
    "spirals_hidden_in_successes": 41,  # of 91 genuine spirals, the old detector caught 50
}

assert RUN_MODE in ("smoke", "full"), "RUN_MODE must be 'smoke' or 'full'"
print("RUN_MODE           :", RUN_MODE)
print("pages this run     :", len(SMOKE_PRINTED_PAGES) if RUN_MODE == "smoke" else "all (~1040)")
print("gold pages         :", GOLD_PRINTED_PAGES)
print("FRESH_START         :", FRESH_START)

## 0 · Environment check

Fail loudly and immediately if the accelerator is not actually attached. A
silent fall back to CPU is the difference between a 2-hour job and a 40-hour
one, and `vision/ocr.py` is written to degrade gracefully to CPU — which is
right for a laptop and wrong here, so we check explicitly.

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
import time

RUN_STARTED = time.time()
print("python  :", platform.python_version())
print("platform:", platform.platform())

subprocess.run(["nvidia-smi"], check=False)

import torch  # noqa: E402

print("\ntorch          :", torch.__version__)
print("cuda available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda version   :", torch.version.cuda)
    print("gpu count      :", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  gpu[{i}]       : {name} ({total:.1f} GB)")
else:
    print("\n*** WARNING: no GPU visible. Nougat on CPU is an overnight run. ***")
    print("*** Fix: notebook sidebar -> Accelerator -> GPU T4 x2, then re-run. ***")

usage = shutil.disk_usage(WORK_DIR)
print(f"\n/kaggle/working free: {usage.free / 1e9:.1f} GB of {usage.total / 1e9:.1f} GB")
print("(corpus needs ~1 GB of page images + ~1 GB of preprocessed interim images)")

## 1 · Get the code

Cloning the public repo is the cleanest way to guarantee the notebook runs
*exactly* the merged code — no copy-paste drift between what is graded on
GitHub and what actually produced these numbers. We record the resolved commit
SHA into the run manifest so every number below is traceable to one commit.

In [ ]:
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
    check=True,
)
os.chdir(REPO_DIR)

# --- keep OCR transcripts OUTSIDE the clone -------------------------------------
# Two problems solved by one symlink:
#   1. Everything under /kaggle/working becomes the notebook's Output. The clone
#      carries ~1 GB of rendered pages plus .git, so run #1's output was a
#      multi-gigabyte download for ~5 MB of text we actually wanted.
#   2. /kaggle/working is the ONLY directory that survives a session. If the
#      transcripts live inside the clone and we delete the clone (or the session
#      dies before the zip cell), a resumable OCR loop has nothing to resume from.
# Pointing data/ocr at /kaggle/working/ocr_live fixes both: text persists and is
# tiny, images stay disposable. `ocr.py` uses a relative Path("data/ocr"), so a
# symlink is transparent to it -- no code change needed.
#
# This ALSO happens to be step 1 of Step 18b's "delete cached .mmd before re-running"
# instruction: data/ocr/ ships committed in git (see .gitignore's un-ignore rule), so the
# clone brings in the OLD pre-repair .mmd files under REPO_DIR/data/ocr -- the rmtree/
# symlink swap below throws them away and starts OCR_LIVE empty, same as it always did.
OCR_LIVE = WORK_DIR + "/ocr_live"
os.makedirs(OCR_LIVE, exist_ok=True)
os.makedirs(os.path.join(REPO_DIR, "data"), exist_ok=True)
_ocr_link = os.path.join(REPO_DIR, "data", "ocr")
if os.path.islink(_ocr_link):
    os.unlink(_ocr_link)
elif os.path.isdir(_ocr_link):
    shutil.rmtree(_ocr_link)
os.symlink(OCR_LIVE, _ocr_link)
print(f"data/ocr -> {OCR_LIVE} (persists across the session; keeps images out of Output)")

REPO_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
REPO_SUBJECT = subprocess.run(
    ["git", "log", "-1", "--pretty=%s"], capture_output=True, text=True, check=True
).stdout.strip()

print("cwd    :", os.getcwd())
print("branch :", REPO_BRANCH)
print("commit :", REPO_COMMIT)
print("subject:", REPO_SUBJECT)

### Dependencies

We deliberately do **not** `pip install -r requirements.lock` here. That lock
pins `torch==2.3.1`, which would fight Kaggle's preinstalled CUDA build of
torch, take ten-plus minutes, and could leave us on a CPU wheel without
noticing. Kaggle's image already ships torch/CUDA, numpy, OpenCV, Pillow,
pydantic and PyYAML.

So we install only what actually differs:

- **`pymupdf`** — `scripts/get_data.sh` renders the PDF with it, and it is not
  in Kaggle's base image.
- **`transformers==4.44.2`** — pinned to match `requirements.lock` exactly, so
  the Nougat generation path here is byte-for-byte the one our lockfile
  describes. (This must happen *before* the first `import transformers`, which
  is why no earlier cell imports it.)
- **`python-Levenshtein` + `nltk`** — see below. These are **hard runtime
  requirements of our own `vision/ocr.py`** that nothing in the repo declares.

### The missing-dependency bug this notebook found

`vision/ocr.py` calls `NougatProcessor.post_process_generation(...)`, and the
very first line of that method in `transformers` is:

```python
requires_backends(self, ["nltk", "levenshtein"])
```

Neither `nltk` nor `python-Levenshtein` appears in `pyproject.toml` or
`requirements.lock`. So **a clean install of this repo cannot run its own OCR
stage** — on Kaggle *or* on a teammate's laptop. It went unnoticed because
Step 11's acceptance bar ("20 pages produce sane LaTeX") was never actually
executed: nobody had the 1.4 GB Nougat weights locally.

The first smoke run died here with `ImportError: NougatTokenizerFast requires
the python-Levenshtein library`, after 22 minutes — which is precisely the
15-minutes-not-3-hours trade the smoke phase exists to make.

Installing them here unblocks the run. The *proper* fix is to declare them in
`pyproject.toml` + `requirements.lock` so the repo is self-consistent; that is
a shared-dependency change and is flagged for the team rather than smuggled in.

(`nltk`'s `words` corpus is *not* needed — the one call that uses it sits inside
`try/except LookupError` and degrades to appending a space. We still fetch it
best-effort so behaviour matches a fully provisioned environment.)

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pymupdf",
     "transformers==" + TRANSFORMERS_PIN,
     "python-Levenshtein",
     "nltk"],
    check=True,
)

# Best effort only: the corpus is optional (see the note above), so a failure to
# fetch it must not take down a multi-hour GPU job.
try:
    import nltk

    nltk.download("words", quiet=True)
    print("nltk 'words' corpus: available")
except Exception as exc:
    print("nltk 'words' corpus unavailable (harmless, falls back):", exc)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))

import cv2  # noqa: E402
import numpy  # noqa: E402
import pydantic  # noqa: E402
import pymupdf  # noqa: E402
import transformers  # noqa: E402
from transformers import NougatProcessor, VisionEncoderDecoderModel  # noqa: E402,F401
from transformers.utils import is_levenshtein_available, is_nltk_available  # noqa: E402

from doc_agent.vision.ocr import NOUGAT_REVISION  # noqa: E402

VERSIONS = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "pymupdf": pymupdf.__doc__.split()[1] if pymupdf.__doc__ else "unknown",
    "opencv": cv2.__version__,
    "numpy": numpy.__version__,
    "pydantic": pydantic.__version__,
}
for k, v in VERSIONS.items():
    print(f"{k:<13}: {v}")

assert transformers.__version__ == TRANSFORMERS_PIN, "transformers pin did not take effect"

# Pre-flight the exact backend check that killed run #1, using transformers' own
# predicates -- so a missing backend fails HERE in seconds rather than after
# 20 minutes of rendering and a model download.
print("\nNougat post-processing backends:")
print("  python-Levenshtein :", is_levenshtein_available())
print("  nltk               :", is_nltk_available())
assert is_levenshtein_available(), "python-Levenshtein missing - post_process_generation will fail"
assert is_nltk_available(), "nltk missing - post_process_generation will fail"

# Exercise the real post-processing path on a dummy string. This is the cheapest
# possible end-to-end proof that the reader can finish a page. We pull the same
# pinned revision vision/ocr.py uses (bandit B615: from_pretrained without a
# revision is a supply-chain risk), so this probe warms the cache the real
# reader will hit rather than downloading a second copy.
_proc = NougatProcessor.from_pretrained("facebook/nougat-base", revision=NOUGAT_REVISION)
_probe = _proc.post_process_generation("6.1.8 \\Gamma(\\tfrac12)=\\pi^{1/2}", fix_markdown=False)
print("\npost_process_generation smoke:", repr(_probe[:60]))
print("Nougat pipeline imports and post-processes cleanly -> safe to proceed.")

## 2 · Get the corpus

`scripts/get_data.sh` (Step 3, S3) is the reproducible corpus path — the same
script a grader runs on a clean machine. It downloads the A&S PDF, verifies its
**sha256** against a recorded value (a truncated download is the classic silent
corpus bug), renders every page to 300-dpi grayscale PNG, and then *spot-checks
the printed-to-PDF page offset against the PDF's own text layer* — the check
that stops a whole sprint from citing pages 32 off.

We render the **whole book even in smoke mode**, because `LIMIT=20` would render
the first 20 *PDF* pages — which are all front matter (`as_f*`), and
`ingest/loader.py` drops those. A limited render would hand us **zero content
pages**. Rendering everything and then subsetting the `Page` list is both
simpler and correct.

**Measured on run #1: 20.1 minutes** for 1082 pages -> 0.94 GB (my pre-run guess
of "4-5 minutes" was wrong by 4x, so treat this as the real fixed cost of any
run, smoke included). The offset spot-check passed **6/6 with the control at
0/6** — the printed-to-PDF mapping is confirmed, not assumed.

It is idempotent: pages already present are skipped.

In [ ]:
t0 = time.time()
subprocess.run(["bash", "scripts/get_data.sh"], check=True)
T_RENDER = time.time() - t0
print("\n[timing] corpus fetch + render: %.1f min" % (T_RENDER / 60))

## 3 · Resume support (only matters if a previous run timed out)

`ocr.transcribe` skips any page whose `.mmd` already exists, so a re-run
continues instead of restarting. `/kaggle/working` does **not** persist across
notebook versions, though — so to actually resume you must feed the previous
run's output back in:

1. previous run finished/timed out -> its `/kaggle/working` became that
   version's **Output**
2. in this notebook: **Add Input -> Notebook Output -> (previous version)**
3. re-run — the cell below copies any `.mmd` it finds under `/kaggle/input`
   into `data/ocr/`, and OCR picks up from there

On a first run this reports `0 seeded`, which is expected.

In [ ]:
if not FRESH_START:
    import glob

    os.makedirs("data/ocr", exist_ok=True)
    seeded = 0
    for src in glob.glob("/kaggle/input/**/*.mmd", recursive=True):
        dst = os.path.join("data/ocr", os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            seeded += 1
    for name in ("meta.jsonl", "failures.json"):
        for src in glob.glob("/kaggle/input/**/" + name, recursive=True):
            dst = os.path.join("data/ocr", name)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)

    print(f"seeded {seeded} cached .mmd transcripts from /kaggle/input")
    print(f"data/ocr now holds {len(glob.glob('data/ocr/*.mmd'))} .mmd files")
else:
    # Step 18b: every previously produced .mmd was written by the pre-repair reader, so
    # resuming from ANY prior run's output (Step 16's or an earlier Step 18b attempt's)
    # would silently keep the bug's output for whatever pages it already "succeeded" on.
    # FRESH_START=True is the default for exactly this reason -- see the CONFIG cell.
    import glob

    os.makedirs("data/ocr", exist_ok=True)
    n_attached = len(glob.glob("/kaggle/input/**/*.mmd", recursive=True))
    print("FRESH_START=True: skipping the /kaggle/input resume-seed.")
    if n_attached:
        print(
            f"  ({n_attached} .mmd files ARE attached as input but were NOT copied -- "
            "set FRESH_START=False only for a same-run resume.)"
        )
    print(f"data/ocr now holds {len(glob.glob('data/ocr/*.mmd'))} .mmd files")

## 4 · Ingest -> clean -> layout

Three stages, all CPU-bound except the optional table-transformer pass:

- **`loader.load_pages`** globs `data/pages/as_p*.png`, drops front matter and
  blank versos (measured ink-variance threshold), and sets `Page.doc_id` to the
  **chapter id** — `doc_id` *is* the chapter for us (`summary.md` §3f), which is
  what makes the train/val/test split leak-proof by construction.
- **`preprocess.run`** deskews only pages measurably crooked (a dead zone
  avoids resampling ~94% of pages for nothing), lightly denoises with a
  bilateral filter, and applies CLAHE. Output stays **grayscale** — hard
  binarisation thickens the thin sub/superscript strokes this corpus is made of.
- **`layout.detect`** finds the two-column gutter, segments blocks within each
  column, classifies them (text/table/figure/heading), and emits regions in
  **reading order** — left column fully, then right. Reading order is not
  cosmetic: our char-F1 uses a longest-common-*subsequence*, so scrambled
  regions score ~0.42 where a bag-of-characters metric would wrongly say 1.00.

**Smoke mode subsets the page list here**, before preprocess and layout — not
via `ocr.transcribe`'s `limit_pages` guard. Both give 20 pages of OCR, but
subsetting early also skips ~1020 pages of preprocessing and layout, turning a
~25-minute smoke run into a ~2-minute one.

In [ ]:
from doc_agent import config
from doc_agent.eval import metrics
from doc_agent.ingest import loader, preprocess
from doc_agent.vision import layout
from doc_agent.vision import ocr as ocr_mod

cfg = config.load()
print("config device:", cfg["device"], "| ocr model:", cfg["ocr"]["model"])
print("layout model :", cfg["layout"]["model"])

t0 = time.time()
all_pages = loader.load_pages(cfg)
T_LOAD = time.time() - t0
print(f"\n[loader] {len(all_pages)} content pages (front matter + blank versos dropped) in {T_LOAD:.1f}s")

smoke_ids = {f"as_p{p:04d}" for p in SMOKE_PRINTED_PAGES}
gold_ids = {f"as_p{p:04d}" for p in GOLD_PRINTED_PAGES}

if RUN_MODE == "smoke":
    pages = [p for p in all_pages if p.id in smoke_ids]
    missing = smoke_ids - {p.id for p in pages}
    assert not missing, f"smoke pages missing from the corpus: {sorted(missing)}"
    print(f"[smoke] subset to {len(pages)} pages")
else:
    pages = all_pages
    print(f"[full] processing all {len(pages)} pages")

t0 = time.time()
pages = preprocess.run(pages, cfg)
T_PREPROCESS = time.time() - t0
print(f"[preprocess] {len(pages)} pages in {T_PREPROCESS / 60:.1f} min")

t0 = time.time()
regions = layout.detect(pages, cfg)
T_LAYOUT = time.time() - t0
kinds = {}
for r in regions:
    kinds[r.kind] = kinds.get(r.kind, 0) + 1
print(f"[layout] {len(regions)} regions from {len(pages)} pages in {T_LAYOUT / 60:.1f} min")
print("[layout] region kinds:", dict(sorted(kinds.items())))
print(f"[layout] regions/page: {len(regions) / max(len(pages), 1):.1f}")

## 5 · OCR — the GPU job

`ocr.transcribe` groups regions **by page** and runs Nougat **once per page**,
not once per region. That is not a shortcut: `facebook/nougat-base` was trained
to read a whole page and emit `.mmd`, so per-page inference is both the faster
path and the more accurate one (it keeps full-page context). The page transcript
is then split back across that page's regions using Step 10's reading order.

Three safeguards are already built into the reader (Step 11):

1. **Resumable** — a page whose `.mmd` exists is not re-inferred.
2. **Token cap** (`max_new_tokens=1536`) — well under Nougat's 4096 limit,
   because a decoder running to the true limit on a dense table is precisely the
   degeneration case, and paying for it on a page we will discard is waste.
3. **Degeneration detector** — a repeated-n-gram tail marks the page **failed**
   and writes it to `data/ocr/failures.json` rather than writing garbage.

We OCR the gated smoke subset first in **both** modes (free, thanks to
resumability), sanity-check it, and only then let the full book run.

In [ ]:
smoke_regions = [r for r in regions if r.page_id in smoke_ids]
print(f"OCR pass 1 (gated subset): {len(smoke_regions)} regions across "
      f"{len({r.page_id for r in smoke_regions})} pages")

t0 = time.time()
smoke_chunks = ocr_mod.transcribe(smoke_regions, cfg)
T_OCR_SMOKE = time.time() - t0
n_smoke_pages = len(glob.glob("data/ocr/*.mmd"))
print(f"\n[timing] gated subset: {T_OCR_SMOKE / 60:.1f} min ({len(smoke_chunks)} chunks, {n_smoke_pages} .mmd on disk)")

## 6 · The sanity gate

This is the checkpoint the whole two-phase protocol exists for.

### Why this gate was rewritten

The first version of this gate **passed run #2** — and run #2 was 35% garbage.
It only checked plumbing: "did we get text, is it long enough, does it contain a
backslash". All true of output where 7 of 20 pages were unusable, `ocr_conf` was
a hardcoded constant, and `exact_formula_match` was structurally incapable of
returning anything but zero. A gate that says **PASSED — safe to run the full
book** over that is worse than no gate, because it converts "unverified" into
"verified" without doing any verifying.

The gold pages were sitting right there, unused. So now the gate reads them.

### What it checks now

**Hard checks — these raise and abort the run:**

1. No gold page went missing *without a documented reason*. A gold page that fails
   with a recorded reason (`[MISSING_PAGE_POST]`, repetition-degeneration, ...) is
   not a bug — that is the honest "before" finding this step exists to report
   (plan.md point 3). An UNEXPLAINED absence (no transcript, no failure record —
   something crashed or was silently dropped) is the real bug, and that is what
   actually aborts here. **Run #3 got this wrong**: it hard-failed on printed 243
   and 255 for being documented baseline failures, which is exactly backwards —
   those are two of the most informative results the whole run produced.
2. Chunk ids are well formed (`chapter|page|region[|formula]`).
3. `extract_formulas` parses a non-zero number of formulas *from the reader's own
   output* on at least one gold page. Zero across the board is the signature of a
   parser/format mismatch rather than a bad reader — the exact bug that made run
   #2's exact-match meaningless.
4. **fp16 regression check**, computed only over gold pages where BOTH an fp16
   result and an fp32 reference exist (a page either side failed on is not a valid
   comparison point). If half precision degrades quality beyond tolerance on a page
   that genuinely succeeded, that is a real finding and the run stops.

**Reported loudly, but not gating** — because these are *measurements*, and a gate
that fails on a low baseline would be a gate that refuses to let us measure a bad
baseline, which is the whole point of the step:

- per-page char-F1 and exact-formula-match against gold
- the failure count **broken down by reason**, using the repaired detector

In [ ]:
GATE_MIN_CHARS = 200

mmd_files = sorted(glob.glob("data/ocr/*.mmd"))
gate_pages = [f for f in mmd_files if os.path.basename(f)[:-4] in smoke_ids]

lengths = {}
for f in gate_pages:
    lengths[os.path.basename(f)[:-4]] = len(open(f, encoding="utf-8").read())

failures = []
if os.path.exists(ocr_mod.FAILURES_PATH):
    failures = json.load(open(ocr_mod.FAILURES_PATH, encoding="utf-8"))
failed_ids = {row["page_id"] for row in failures}
failed_ids_reason = {row["page_id"]: row.get("reason", "?") for row in failures}

produced = [pid for pid in smoke_ids if pid in lengths]
substantial = [pid for pid in produced if lengths[pid] >= GATE_MIN_CHARS]

print(f"gated pages requested : {len(smoke_ids)}")
print(f"transcripts produced  : {len(produced)}")
print(f"marked degenerate     : {len(failed_ids & smoke_ids)}  {sorted(failed_ids & smoke_ids)}")
print(f"length >= {GATE_MIN_CHARS} chars   : {len(substantial)}")
if lengths:
    vals = sorted(lengths.values())
    print(f"transcript chars      : min {vals[0]} / median {vals[len(vals) // 2]} / max {vals[-1]}")

sample_id = "as_p0255" if "as_p0255" in lengths else (produced[0] if produced else None)
if sample_id:
    text = open(f"data/ocr/{sample_id}.mmd", encoding="utf-8").read()
    print("\n" + "=" * 78)
    print(f"SAMPLE TRANSCRIPT — {sample_id} (printed page {sample_id[4:].lstrip('0')})")
    print("=" * 78)
    print(text[:1500])
    print(f"... [{len(text)} chars total]")

print("\n" + "=" * 78)
print("SAMPLE CHUNK IDS (citation anchors)")
print("=" * 78)
for c in smoke_chunks[:10]:
    print(f"  {c.id:<45}  {len(c.text):3d} chars")
with_formula = [c for c in smoke_chunks if c.id.count("|") >= 3]
print(f"\nchunks carrying a formula id: {len(with_formula)} of {len(smoke_chunks)}")
for c in with_formula[:5]:
    print(f"  {c.id}")

# --- honest failure breakdown, using the repaired detector ---------------------------
print("\n" + "=" * 78)
print("FAILURE BREAKDOWN (form Section 5 asks for this explicitly)")
print("=" * 78)
by_reason = {}
for row in failures:
    by_reason[row.get("reason", "?")] = by_reason.get(row.get("reason", "?"), 0) + 1
print(f"pages attempted : {len(pages)}")
print(f"failed          : {len(failures)}  ({100.0 * len(failures) / max(len(pages), 1):.0f}%)")
for reason, count in sorted(by_reason.items(), key=lambda kv: -kv[1]):
    print(f"   {reason:<32} {count}")
if not failures:
    print("   (none)")

# --- quality against the gold pages ---------------------------------------------------
# fp32 baseline measured in Step 16 run #2 on this exact 20-page subset. fp16 is a speed
# change and must not cost accuracy; if it does, that is a finding worth stopping for.
FP32_BASELINE_CHAR_F1 = {"as_p0243": 0.1984, "as_p0255": 0.2400, "as_p0360": 0.4130}
FP16_TOLERANCE = 0.05

gold_labels = {}
for _line in open("grading_kit/labels.jsonl", encoding="utf-8"):
    _line = _line.strip()
    if _line:
        _row = json.loads(_line)
        gold_labels[_row["page_id"]] = _row["text"]

print("\n" + "=" * 78)
print("QUALITY vs GOLD  (measurement, not a pass/fail bar)")
print("=" * 78)
print(f"{'page':<10} {'char-F1':>9} {'fp32 ref':>9} {'delta':>11} {'gold form':>10} {'pred form':>10}")
print("-" * 78)
gate_quality = []
unexplained_missing = []  # missing with NO failure record -> real bug, not baseline quality
for pid in sorted(gold_ids):
    mmd = f"data/ocr/{pid}.mmd"
    if pid not in gold_labels or not os.path.exists(mmd):
        reason = failed_ids_reason.get(pid)
        if reason is not None:
            # A documented baseline failure on a gold page IS the honest finding this step
            # exists to report (plan.md Step 16 point 3, carried into Step 18b: "log
            # every page that failed or degenerated ... count them, report this
            # honestly"). It is not a pipeline bug, so it must not abort the run --
            # Step 16 run #3 aborted here on exactly this, over-specified relative to
            # what this gate actually asks for.
            print(f"{pid:<10}  no transcript -- documented baseline failure: {reason}")
        else:
            print(f"{pid:<10}  no transcript -- NO failure record either (unexplained)")
            unexplained_missing.append(pid)
        continue
    _pred = open(mmd, encoding="utf-8").read()
    _gold = gold_labels[pid]
    f1 = metrics.ocr_f1(_pred, _gold)
    ref = FP32_BASELINE_CHAR_F1.get(pid)
    n_gold_f = len(metrics.extract_formulas(_gold))
    n_pred_f = len(metrics.extract_formulas(_pred))
    delta = (f1 - ref) if ref is not None else None
    ref_s = f"{ref:.4f}" if ref else "-"
    delta_s = f"{delta:+.4f}" if delta is not None else "-"
    print(f"{pid:<10} {f1:9.4f} {ref_s:>9} {delta_s:>11} {n_gold_f:10d} {n_pred_f:10d}")
    gate_quality.append({"page_id": pid, "f1": f1, "ref": ref,
                         "n_gold_f": n_gold_f, "n_pred_f": n_pred_f})
print("-" * 78)

print("\n" + "=" * 78)
print("GATE RESULTS")
print("=" * 78)

regressed = [q for q in gate_quality
             if q["ref"] is not None and q["f1"] < q["ref"] - FP16_TOLERANCE]
parsed_any_formula = any(q["n_pred_f"] > 0 for q in gate_quality if q["n_gold_f"] > 0)
gold_with_formulas = [q for q in gate_quality if q["n_gold_f"] > 0]

checks = [
    # NOT gated on "every gold page produced a transcript". A pretrained baseline
    # failing on a hard gold page (a repetition spiral, a [MISSING_PAGE] marker) is
    # the honest "before" finding this step exists to report (plan.md point 3), not a
    # pipeline defect -- printed above with its documented reason either way. Only an
    # UNEXPLAINED absence (no transcript AND no failure record: something crashed or
    # was silently dropped) is a real bug, and that still aborts the run.
    ("no gold page went missing without a documented reason",
     not unexplained_missing),
    ("chunk ids are well formed (chapter|page|region)",
     len(smoke_chunks) > 0 and all(c.id.count("|") >= 2 for c in smoke_chunks)),
    ("extract_formulas parses the reader's OWN output (not just gold)",
     parsed_any_formula or not gold_with_formulas),
    (f"fp16 did not regress char-F1 vs the fp32 baseline (tol {FP16_TOLERANCE:.2f}, "
     f"{len(gate_quality)} page(s) comparable)",
     not regressed),
]
for label, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}")

GATE_PASSED = all(ok for _, ok in checks)
print(f"\nGATE: {'PASSED' if GATE_PASSED else 'FAILED'}")
print("NOTE: passing means the pipeline is SOUND, not that quality is GOOD.")
print(f"      {len(gold_ids) - len(gate_quality)} of {len(gold_ids)} gold pages failed "
      "under the pretrained baseline (see above) --")
print("      that is real, reportable baseline weakness, not a reason to distrust this run.")
if not GATE_PASSED:
    if unexplained_missing:
        print("\nunexplained missing pages (investigate before proceeding):", unexplained_missing)
    if regressed:
        print("\nfp16 regression detail:")
        for q in regressed:
            print(f"  {q['page_id']}: {q['f1']:.4f} vs fp32 {q['ref']:.4f} ({q['f1'] - q['ref']:+.4f})")
    raise RuntimeError(
        f"Sanity gate failed on the {len(smoke_ids)}-page subset. Do NOT launch the full "
        "run -- fix the cause first."
    )

## 7 · The full book

Only reached with a green gate. In `"smoke"` mode this cell is a no-op, so the
same notebook is safe to run either way.

Expect roughly **1.5–3 hours** on a T4 for ~1040 pages. `ocr.transcribe` logs
progress every 50 pages; the pages already done in the gated pass are skipped.

In [ ]:
if RUN_MODE == "full":
    print(f"Starting full-book OCR over {len(regions)} regions / "
          f"{len({r.page_id for r in regions})} pages.")
    print("Pages already transcribed are skipped. Progress logs every 50 pages.\n")
    t0 = time.time()
    chunks = ocr_mod.transcribe(regions, cfg)
    T_OCR_FULL = time.time() - t0
    print(f"\n[timing] full-book OCR: {T_OCR_FULL / 3600:.2f} h")
else:
    chunks = smoke_chunks
    T_OCR_FULL = 0.0
    print("RUN_MODE='smoke' — full-book pass skipped.")
    print("Review the sample transcript above, then set RUN_MODE='full' and re-push.")

mmd_files = sorted(glob.glob("data/ocr/*.mmd"))
print(f"\ntotal transcripts on disk: {len(mmd_files)}")
print(f"total chunks produced    : {len(chunks)}")

## 8 · Honest failure accounting

`plan.md` Step 16 point 3 and form §5 both ask for this explicitly: **count the
pages that failed and report them**, rather than quietly shipping a smaller
corpus and implying full coverage. Nougat's repetition-degeneration on dense
numeric tables is a known, documented weakness of the baseline model — it is
exactly the kind of failure the Sprint-4 fine-tune is meant to reduce, so having
the number here makes the before/after comparison meaningful.

In [ ]:
import csv

failures = []
if os.path.exists(ocr_mod.FAILURES_PATH):
    failures = json.load(open(ocr_mod.FAILURES_PATH, encoding="utf-8"))

pages_attempted = len(pages)
pages_ok = len(mmd_files)
pages_failed = len(failures)

print(f"pages attempted this run : {pages_attempted}")
print(f"transcripts on disk      : {pages_ok}")
print(f"degenerate / failed      : {pages_failed}")
if pages_attempted:
    print(f"failure rate             : {100.0 * pages_failed / pages_attempted:.2f}%")

if failures:
    print("\nfailed pages:")
    for row in failures:
        print(f"  {row.get('page_id')}  reason={row.get('reason')}  at={row.get('detected_at')}")
else:
    print("\nNo degenerate pages recorded.")

# Per-page coverage table -> ocr_coverage.csv (csv imported at cell top)
coverage_rows = []
by_id = {p.id: p for p in pages}
for f in mmd_files:
    pid = os.path.basename(f)[:-4]
    text = open(f, encoding="utf-8").read()
    coverage_rows.append({
        "page_id": pid,
        "doc_id": by_id[pid].doc_id if pid in by_id else "",
        "n_chars": len(text),
        "n_words": len(text.split()),
        "status": "failed" if pid in {r["page_id"] for r in failures} else "ok",
    })

TOTAL_WORDS_OUR_OCR = sum(r["n_words"] for r in coverage_rows)
TOTAL_CHARS_OUR_OCR = sum(r["n_chars"] for r in coverage_rows)
print(f"\nwords transcribed by OUR OCR : {TOTAL_WORDS_OUR_OCR}")
print(f"chars transcribed by OUR OCR : {TOTAL_CHARS_OUR_OCR}")
if RUN_MODE == "full":
    floor = 60000
    print(f"configs/task.yaml floor      : {floor} words")
    met = "YES" if TOTAL_WORDS_OUR_OCR >= floor else "NO — investigate before Step 30"
    print(f"floor met                    : {met}")
    print("(data/validate.py enforces this from OUR OCR, never the archive text layer)")

# --- plan.md Step 18b's explicit gate -----------------------------------------------
# "no-output rate" = pages attempted this run that never produced a usable transcript,
# i.e. the SAME pages_failed/pages_attempted ratio above, restated as the pass/fail
# decision plan.md actually asks for. <=25% -> proceed to Step 19 with the 105-page
# training set unchanged. >25% -> the annotation budget question reopens, evidence-based
# (plan.md is explicit this must be a measured decision, not an assumption).
if RUN_MODE == "full" and pages_attempted:
    no_output_rate = pages_failed / pages_attempted
    print("\n" + "=" * 78)
    print("STEP 18B GATE — no-output rate")
    print("=" * 78)
    print(f"no-output rate this run : {100 * no_output_rate:.1f}%  "
          f"(BEFORE repair, naive: {100 * BEFORE_REPAIR['naive_failure_rate']:.1f}%)")
    print("gate threshold          : <= 25%")
    if no_output_rate <= 0.25:
        print("GATE: PASSED — proceed to Step 19 with the 105-page training set unchanged.")
    else:
        print("GATE: FAILED — reopen the annotation-budget question (plan.md Step 18b),")
        print("      stratified by whichever failure mode is still dominant below.")


In [ ]:
# --- PDF text-layer coverage: the diagnostic that found the Step 18b bug -----------------
# Step 16's own gate (section 6) only asks "did each page clear 200 chars and pass the
# degeneracy check" -- exactly the measure that hid this bug, since a page that stops
# after the first third of its content still clears 200 chars easily. The real signal is
# coverage against the PDF's OWN text layer (pymupdf's get_text("text")), used here purely
# as a ground-truth word count for "how much of the page did we actually capture" -- never
# fed into the reader or the corpus itself (the text layer is untrustworthy for CONTENT,
# summary.md 7a, but a word count off it is fine as a reference).
import pymupdf

_pdf_doc = pymupdf.open(PDF_PATH)


def _pdf_word_count(printed_page: int) -> int | None:
    pdf_index = printed_page + FRONT_MATTER_OFFSET - 1  # 0-based
    if pdf_index < 0 or pdf_index >= _pdf_doc.page_count:
        return None
    return len(_pdf_doc.load_page(pdf_index).get_text("text").split())


per_page_coverage = []
zero_coverage_pages = []
for f in mmd_files:
    pid = os.path.basename(f)[:-4]
    if not pid.startswith("as_p"):
        continue
    pdf_words = _pdf_word_count(int(pid[4:]))
    if not pdf_words:
        continue
    ocr_words = len(open(f, encoding="utf-8").read().split())
    ratio = ocr_words / pdf_words
    per_page_coverage.append(ratio)
    if ratio == 0:
        zero_coverage_pages.append(pid)

per_page_coverage.sort()
n_cov = len(per_page_coverage)
median_coverage = per_page_coverage[n_cov // 2] if n_cov else 0.0
mean_coverage = sum(per_page_coverage) / n_cov if n_cov else 0.0

# Book-wide: OUR total transcribed words (this run) vs the PDF's total content-page words
# (all pages after the front matter), same ratio Elias's diagnostic measured at 15.1%.
book_wide_pdf_words = sum(
    len(_pdf_doc.load_page(i).get_text("text").split())
    for i in range(FRONT_MATTER_OFFSET, _pdf_doc.page_count)
)
book_wide_coverage = TOTAL_WORDS_OUR_OCR / book_wide_pdf_words if book_wide_pdf_words else 0.0

print("=" * 78)
print("PDF TEXT-LAYER COVERAGE  (the Step 18b diagnostic)")
print("=" * 78)
print(f"pages compared            : {n_cov}")
print(f"median per-page coverage  : {100 * median_coverage:5.1f}%   "
      f"(BEFORE repair: {100 * BEFORE_REPAIR['median_page_coverage_vs_pdf_text_layer']:5.1f}%)")
print(f"mean per-page coverage    : {100 * mean_coverage:5.1f}%")
print(f"book-wide word coverage   : {100 * book_wide_coverage:5.1f}%   "
      f"(BEFORE repair: {100 * BEFORE_REPAIR['book_wide_word_coverage_vs_pdf_text_layer']:5.1f}%)")
shown = zero_coverage_pages[:10] if zero_coverage_pages else ""
print(f"pages with zero coverage  : {len(zero_coverage_pages)}  {shown}")
print()
print("If median/book-wide coverage did not move up substantially from the BEFORE numbers,")
print("the repair did not fix the recall problem and defect 4 (early stopping on the")
print("two-column layout, a genuine model limitation per plan.md Step 18b) may dominate --")
print("that is a real, reportable finding, not a reason to distrust this measurement.")


## 9 · The BEFORE numbers — baseline quality on the 3 gold pages

This is the deliverable that Sprint 4 is measured against.

We report **two** metrics because either one alone is misleading
(`summary.md` §4e):

- **char-level F1** (`metrics.ocr_f1`) — longest-common-*subsequence* based, so
  it is sensitive to reading order. Both sides are normalised first
  (`normalize_latex`: Unicode NFC, `\tfrac`->`\frac`, `\left(`->`(`, spacing
  macros dropped) so we score *reading accuracy*, not notation style.
- **exact formula-match rate** (`metrics.exact_formula_match`) — the fraction of
  the page's numbered formulas reproduced **exactly**, aligned by A&S formula id.

Why both: char-F1 flatters mathematics badly. S3 measured this while building
Step 12 — corrupting 5% of characters on the real p.255 gold still reads
**0.955 char-F1** while breaking **every single formula** on the page. A number
that high next to an exact-match of 0.000 is why reporting one metric alone
would be dishonest.

**On aggregation:** `exact_formula_match` returns `0.0` for a page with no
numbered formulas — printed 243 is a pure numeric table, so it has none, and a
`0.0` there means "nothing to measure", not "read it wrong". Averaging that in
blindly would drag the score down for a reason unrelated to OCR quality, so we
weight each page by its formula count and report both aggregates.

In [ ]:
labels = {}
for line in open("grading_kit/labels.jsonl", encoding="utf-8"):
    line = line.strip()
    if line:
        row = json.loads(line)
        labels[row["page_id"]] = row["text"]

per_page = []
for printed in GOLD_PRINTED_PAGES:
    pid = f"as_p{printed:04d}"
    mmd = f"data/ocr/{pid}.mmd"
    if pid not in labels:
        print(f"SKIP {pid} — no gold label")
        continue
    if not os.path.exists(mmd):
        print(f"SKIP {pid} — no transcript produced")
        continue

    gold = labels[pid]
    pred = open(mmd, encoding="utf-8").read()
    gold_formulas = metrics.extract_formulas(gold)
    pred_formulas = metrics.extract_formulas(pred)

    # Precision/recall behind char-F1 (same LCS decomposition ocr_f1 uses internally,
    # exposed here because plan.md Step 18b's whole diagnosis rests on this split: Step
    # 16's as_p0360 was precision 0.813 / recall 0.280 -- what it transcribed was
    # accurate, it simply stopped early. F1 alone hides which of those moved.
    _p_norm, _g_norm = metrics.normalize_latex(pred), metrics.normalize_latex(gold)
    _overlap = metrics._lcs_length(_p_norm, _g_norm)
    _precision = _overlap / len(_p_norm) if _p_norm else 0.0
    _recall = _overlap / len(_g_norm) if _g_norm else 0.0

    per_page.append({
        "page_id": pid,
        "printed_page": printed,
        "char_f1": round(metrics.ocr_f1(pred, gold), 4),
        "precision": round(_precision, 4),
        "recall": round(_recall, 4),
        "exact_formula_match": round(metrics.exact_formula_match(pred, gold), 4),
        "n_gold_formulas": len(gold_formulas),
        "n_pred_formulas": len(pred_formulas),
        "gold_chars": len(gold),
        "pred_chars": len(pred),
        "degenerate": pid in {r["page_id"] for r in failures},
    })

print("=" * 92)
print("STEP 18B AFTER — same pretrained facebook/nougat-base, repaired inference path")
print("=" * 92)
print(f"{'page':<10} {'printed':>8} {'char-F1':>10} {'precision':>10} {'recall':>10} "
      f"{'exact-form':>12} {'gold-form':>10} {'pred-form':>10}")
print("-" * 92)
for r in per_page:
    print(f"{r['page_id']:<10} {r['printed_page']:>8d} {r['char_f1']:>10.4f} "
          f"{r['precision']:>10.4f} {r['recall']:>10.4f} {r['exact_formula_match']:>12.4f} "
          f"{r['n_gold_formulas']:>10d} {r['n_pred_formulas']:>10d}")
print("-" * 92)

_p360 = next((r for r in per_page if r["page_id"] == "as_p0360"), None)
if _p360:
    print("\nas_p0360 precision/recall — the flagship Step 18b diagnostic page:")
    print(f"  BEFORE repair : precision {BEFORE_REPAIR['as_p0360_precision']:.3f} / "
          f"recall {BEFORE_REPAIR['as_p0360_recall']:.3f}")
    print(f"  AFTER  repair : precision {_p360['precision']:.3f} / recall {_p360['recall']:.3f}")
    print("  (plan.md Step 18b: as_p0360's 0.4166 char-F1 was 'part OCR quality and part")
    print("   scoring artifact' pre-repair -- this is the re-measurement it asked for.)")

BASELINE = {}
if per_page:
    n = len(per_page)
    mean_f1 = sum(r["char_f1"] for r in per_page) / n
    mean_precision = sum(r["precision"] for r in per_page) / n
    mean_recall = sum(r["recall"] for r in per_page) / n
    mean_efm_unweighted = sum(r["exact_formula_match"] for r in per_page) / n
    total_formulas = sum(r["n_gold_formulas"] for r in per_page)
    weighted_efm = (
        sum(r["exact_formula_match"] * r["n_gold_formulas"] for r in per_page) / total_formulas
        if total_formulas else 0.0
    )
    print(f"\n{'MEAN':<10} {'-':>8} {mean_f1:>10.4f} {mean_precision:>10.4f} "
          f"{mean_recall:>10.4f} {mean_efm_unweighted:>12.4f} {total_formulas:>10d}")
    print(f"{'WEIGHTED':<10} {'-':>8} {'-':>10} {'-':>10} {'-':>10} {weighted_efm:>12.4f}"
          "   <- use this one (weighted by formula count)")
    print("=" * 92)

    BASELINE = {
        "mean_char_f1": round(mean_f1, 4),
        "mean_precision": round(mean_precision, 4),
        "mean_recall": round(mean_recall, 4),
        "mean_exact_formula_match_unweighted": round(mean_efm_unweighted, 4),
        "weighted_exact_formula_match": round(weighted_efm, 4),
        "total_gold_formulas": total_formulas,
        "n_gold_pages": n,
    }

    print("\nsummary.md 4e expects the FINE-TUNED reader to land at")
    print("char-F1 0.88-0.93 and exact-match 0.55-0.75. This baseline is the")
    print("BEFORE column; Step 29 measures AFTER on the full 39-page test set.")
else:
    print("No gold pages transcribed yet — run at least the smoke pass.")

## 10 · Package everything into one download

Two clearly separated top-level folders inside the zip, so unpacking is
unambiguous:

```
mathscholar_step18b_ocr_repair.zip
|
+-- data/ocr/                  <- unzip at the REPO ROOT (committed on purpose; feeds Steps 17 & 30)
|     +-- as_pNNNN.mmd  x ~1040
|     +-- meta.jsonl           per-chunk ocr_confidence + bbox
|     +-- failures.json        the honest degenerate-page list
|
+-- step18b_reports/           <- evidence for the A2 form; keep locally
      +-- baseline_metrics.json
      +-- baseline_metrics.md  paste-ready table
      +-- ocr_coverage.csv     per-page chars/words/status
      +-- run_manifest.json    full reproducibility record
      +-- README.md
```

We ship **only text**, never images: `data/pages/` and `data/interim/` are ~1 GB
each and are regenerated exactly by `bash scripts/get_data.sh`, so shipping them
would cost a gigabyte of download to reproduce something a script already
reproduces deterministically.

In [ ]:
import zipfile

if os.path.isdir(STAGE_DIR):
    shutil.rmtree(STAGE_DIR)
ocr_out = os.path.join(STAGE_DIR, "data", "ocr")
rep_out = os.path.join(STAGE_DIR, "step18b_reports")
os.makedirs(ocr_out, exist_ok=True)
os.makedirs(rep_out, exist_ok=True)

for f in mmd_files:
    shutil.copy2(f, ocr_out)
for name in ("meta.jsonl", "failures.json"):
    p = os.path.join("data/ocr", name)
    if os.path.exists(p):
        shutil.copy2(p, ocr_out)

COVERAGE_METRICS = {
    "median_page_coverage_vs_pdf_text_layer": round(median_coverage, 4),
    "mean_page_coverage_vs_pdf_text_layer": round(mean_coverage, 4),
    "book_wide_word_coverage_vs_pdf_text_layer": round(book_wide_coverage, 4),
    "pages_with_zero_coverage": len(zero_coverage_pages),
    "no_output_rate": round(pages_failed / pages_attempted, 4) if pages_attempted else None,
}

RUN_MANIFEST = {
    "step": "18b",
    "title": "OCR repair re-run (pretrained facebook/nougat-base, inference-path fixes)",
    "owner": "S1 Anuron Maitro (2105037)",
    "run_mode": RUN_MODE,
    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "repo": {"url": REPO_URL, "branch": REPO_BRANCH,
             "commit": REPO_COMMIT, "subject": REPO_SUBJECT},
    "versions": VERSIONS,
    "gpu": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)"),
    "models": {
        "ocr": cfg["ocr"]["model"],
        "ocr_revision": getattr(ocr_mod, "NOUGAT_REVISION", None),
        "ocr_finetuned": False,
        "ocr_max_new_tokens": getattr(ocr_mod, "MAX_NEW_TOKENS", None),
        "ocr_repetition_penalty": getattr(ocr_mod, "REPETITION_PENALTY", None),
        "layout": cfg["layout"]["model"],
        "layout_revision": getattr(layout, "TATR_REVISION", None),
    },
    "config_yaml": cfg,
    "counts": {
        "content_pages_in_corpus": len(all_pages),
        "pages_processed_this_run": len(pages),
        "regions_detected": len(regions),
        "region_kinds": kinds,
        "transcripts_on_disk": len(mmd_files),
        "chunks_produced": len(chunks),
        "pages_failed_degenerate": len(failures),
        "words_from_our_ocr": TOTAL_WORDS_OUR_OCR,
        "chars_from_our_ocr": TOTAL_CHARS_OUR_OCR,
    },
    "timings_seconds": {
        "corpus_render": round(T_RENDER, 1),
        "loader": round(T_LOAD, 1),
        "preprocess": round(T_PREPROCESS, 1),
        "layout": round(T_LAYOUT, 1),
        "ocr_gated_subset": round(T_OCR_SMOKE, 1),
        "ocr_full_book": round(T_OCR_FULL, 1),
        "total": round(time.time() - RUN_STARTED, 1),
    },
    "before_repair_reference": BEFORE_REPAIR,
    "after_repair_coverage": COVERAGE_METRICS,
    "after_repair_gold_page_metrics": BASELINE,
    "after_repair_per_page": per_page,
    "failures": failures,
}

with open(os.path.join(rep_out, "run_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(RUN_MANIFEST, f, indent=2)

with open(os.path.join(rep_out, "baseline_metrics.json"), "w", encoding="utf-8") as f:
    json.dump({"run_mode": RUN_MODE, "before_repair": BEFORE_REPAIR,
               "after_repair_coverage": COVERAGE_METRICS, "after_repair_gold": BASELINE,
               "per_page": per_page, "failures": failures}, f, indent=2)

with open(os.path.join(rep_out, "ocr_coverage.csv"), "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["page_id", "doc_id", "n_chars", "n_words", "status"])
    w.writeheader()
    w.writerows(sorted(coverage_rows, key=lambda r: r["page_id"]))

md = []
md.append("# Step 18b — OCR repair re-run (BEFORE / AFTER numbers)\n")
md.append(f"Reader: `{cfg['ocr']['model']}` (**pretrained, not fine-tuned** -- Step 18b fixes "
          f"the inference path, not the model), revision "
          f"`{getattr(ocr_mod, 'NOUGAT_REVISION', 'n/a')}`.\n")
md.append(f"Run mode: `{RUN_MODE}` · repo commit `{REPO_COMMIT[:12]}` · GPU `{RUN_MANIFEST['gpu']}`.\n")

md.append("\n## The four numbers plan.md Step 18b asks for\n")
md.append("| metric | BEFORE (Step 16) | AFTER (this run) |")
md.append("|---|---:|---:|")
_after_no_output = (f"{100 * COVERAGE_METRICS['no_output_rate']:.1f}%"
                     if COVERAGE_METRICS["no_output_rate"] is not None else "n/a")
md.append(f"| no-output rate | {100 * BEFORE_REPAIR['naive_failure_rate']:.1f}% | {_after_no_output} |")
md.append("| median page coverage vs PDF text layer | "
          f"{100 * BEFORE_REPAIR['median_page_coverage_vs_pdf_text_layer']:.1f}% | "
          f"{100 * COVERAGE_METRICS['median_page_coverage_vs_pdf_text_layer']:.1f}% |")
md.append("| book-wide word coverage vs PDF text layer | "
          f"{100 * BEFORE_REPAIR['book_wide_word_coverage_vs_pdf_text_layer']:.1f}% | "
          f"{100 * COVERAGE_METRICS['book_wide_word_coverage_vs_pdf_text_layer']:.1f}% |")
if BASELINE:
    md.append(f"| mean char-F1 (3 gold pages) | n/a (see per-page) | {BASELINE['mean_char_f1']:.4f} |")
    md.append("| weighted exact-formula-match (3 gold pages) | n/a (see per-page) | "
              f"{BASELINE['weighted_exact_formula_match']:.4f} |")
if any(r["page_id"] == "as_p0360" for r in per_page):
    _p360b = next(r["precision"] for r in per_page if r["page_id"] == "as_p0360")
    _p360r = next(r["recall"] for r in per_page if r["page_id"] == "as_p0360")
    _after_360 = f"{_p360b:.3f} / {_p360r:.3f}"
else:
    _after_360 = "no transcript"
md.append(f"| as_p0360 precision / recall | {BEFORE_REPAIR['as_p0360_precision']:.3f} / "
          f"{BEFORE_REPAIR['as_p0360_recall']:.3f} | {_after_360} |")

md.append("\n## Per-page baseline on the 3 A1 gold pages\n")
md.append("| page | printed | char-F1 | precision | recall | exact-formula | gold formulas | pred formulas |")
md.append("|---|---:|---:|---:|---:|---:|---:|---:|")
for r in per_page:
    md.append(f"| `{r['page_id']}` | {r['printed_page']} | {r['char_f1']:.4f} | "
              f"{r['precision']:.4f} | {r['recall']:.4f} | {r['exact_formula_match']:.4f} | "
              f"{r['n_gold_formulas']} | {r['n_pred_formulas']} |")
if BASELINE:
    md.append(f"\n**Aggregate:** mean char-F1 **{BASELINE['mean_char_f1']:.4f}** · mean "
              f"precision **{BASELINE['mean_precision']:.4f}** · mean recall "
              f"**{BASELINE['mean_recall']:.4f}** · exact-formula-match "
              f"**{BASELINE['weighted_exact_formula_match']:.4f}** (weighted by formula count; "
              f"unweighted {BASELINE['mean_exact_formula_match_unweighted']:.4f} across "
              f"{BASELINE['total_gold_formulas']} formulas on {BASELINE['n_gold_pages']} pages).\n")
md.append("\n## Coverage and honest failures\n")
md.append(f"- content pages in corpus: **{len(all_pages)}**")
md.append(f"- pages processed this run: **{len(pages)}**")
md.append(f"- transcripts produced: **{len(mmd_files)}**")
md.append(f"- degenerate/failed pages: **{len(failures)}**")
md.append(f"- words from OUR OCR: **{TOTAL_WORDS_OUR_OCR}**")
md.append(f"- regions detected: **{len(regions)}** {dict(sorted(kinds.items()))}")
if failures:
    md.append("\nFailed pages: " + ", ".join(f"`{r['page_id']}`" for r in failures))
md.append("\n## Timings\n")
for k, v in RUN_MANIFEST["timings_seconds"].items():
    md.append(f"- {k}: {v / 60:.1f} min")
with open(os.path.join(rep_out, "baseline_metrics.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(md) + "\n")

readme = """# Step 18b output — OCR repair re-run

Produced on Kaggle GPU by `KAGGLE/step18b_ocr_repair/kaggle_step18b_ocr_repair.ipynb`
(MathScholar team 1, S1).

## How to unpack

    unzip mathscholar_step18b_ocr_repair.zip -d /path/to/doc-agent-1

- `data/ocr/` lands in the repo and is **committed on purpose** (see .gitignore's
  un-ignore rule for data/ocr/ -- Steps 17/18 annotation drafts and Step 30's re-OCR
  both read from here, and kb_demo.ipynb needs it present for the grounding gate).
- `step18b_reports/` is evidence for the A2 form. The numbers themselves go into
  `notebooks/kb_demo.ipynb` and `CHANGELOG.md`.

## What is deliberately NOT in this zip

`data/pages/` and `data/interim/` (~1 GB each) are regenerated exactly by
`bash scripts/get_data.sh`, which verifies the source PDF sha256 and the
printed-to-PDF page offset. Shipping them would trade a gigabyte for something
a deterministic script already reproduces.
"""
with open(os.path.join(rep_out, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme)

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for root, _dirs, files in os.walk(STAGE_DIR):
        for name in files:
            full = os.path.join(root, name)
            z.write(full, os.path.relpath(full, STAGE_DIR))

print(f"zip written: {ZIP_PATH}  ({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)")
print("\ncontents:")
print(f"  data/ocr/            {len(os.listdir(ocr_out))} files")
print(f"  step18b_reports/     {len(os.listdir(rep_out))} files")
print("\n--- baseline_metrics.md ---")
print(open(os.path.join(rep_out, "baseline_metrics.md"), encoding="utf-8").read())

### Tidy the Output

Everything under `/kaggle/working` becomes this notebook's downloadable Output.
The clone holds ~1 GB of rendered pages, ~1 GB of preprocessed interim images
and a `.git` directory — none of which anyone should ever download, because
`bash scripts/get_data.sh` regenerates them deterministically (and verifies the
source sha256 while doing it).

Now that the zip is written we delete the clone. The transcripts are safe: they
live in `/kaggle/working/ocr_live`, which the clone only *pointed at*, so
removing the clone cannot touch them.

In [ ]:
os.chdir(WORK_DIR)  # must leave the directory before removing it
try:
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
        print(f"removed the clone: {REPO_DIR}")
except Exception as exc:
    print("could not remove the clone (harmless, Output is just larger):", exc)

print("\nFinal contents of /kaggle/working (this is your Output):")
total = 0
for entry in sorted(os.listdir(WORK_DIR)):
    full = os.path.join(WORK_DIR, entry)
    if os.path.isfile(full):
        size = os.path.getsize(full)
        print(f"  {entry:<45} {size / 1e6:8.1f} MB")
        total += size
    else:
        n = sum(len(fs) for _, _, fs in os.walk(full))
        size = sum(os.path.getsize(os.path.join(r, f))
                   for r, _, fs in os.walk(full) for f in fs)
        print(f"  {entry + '/':<45} {size / 1e6:8.1f} MB  ({n} files)")
        total += size
print(f"  {'TOTAL':<45} {total / 1e6:8.1f} MB")
n_mmd = len(glob.glob(os.path.join(OCR_LIVE, "*.mmd")))
print(f"\nTranscripts preserved in ocr_live/: {n_mmd} .mmd")

## 11 · What to do with this output

### If you just ran `RUN_MODE = "smoke"`

1. Read the **sample transcript** in section 6 — does it look like the real
   page? Real LaTeX, sane structure, no repeated-token loop?
2. Check the **chunk ids** — `ch06_gamma|as_p0255|r00|6.1.8` shaped, with
   formula ids appearing on formula pages.
3. Look at the **coverage numbers** in section 8 (PDF text-layer coverage) and
   the **gold-page numbers** in section 9 (char-F1 / precision / recall). A
   pretrained (not yet fine-tuned) reader on a 1964 scan should land clearly
   *below* the 0.88–0.93 char-F1 band `summary.md` §4e predicts for the
   fine-tuned model — that gap is the thing Sprint 4 has to close. What matters
   for Step 18b specifically is that median/book-wide coverage moved up
   substantially from the BEFORE numbers printed alongside them; a baseline
   that is already at 0.95 would mean something is wrong with the measurement,
   not that we are done.
4. If all of the above look right: set `RUN_MODE = "full"` and push again.

### If you just ran `RUN_MODE = "full"`

1. Download `mathscholar_step18b_ocr_repair.zip` from the notebook's **Output**
   panel (or `kaggle kernels output`).
2. Unzip it at the repo root so `data/ocr/` is populated (committed, not
   gitignored — see the un-ignore rule at the top of `.gitignore`).
3. Read the **STEP 18B GATE** printout in section 8: no-output rate <=25% means
   proceed to Step 19 with the 105-page training set unchanged (plan.md's own
   decision, re-confirmed here with real post-repair data); >25% means the
   annotation-budget question reopens, stratified by whichever failure mode is
   still dominant in `failures.json`.
4. Copy the numbers from `step18b_reports/baseline_metrics.md` (all four:
   no-output rate, PDF text-layer coverage, char-F1, exact-formula-match) into
   `notebooks/kb_demo.ipynb` (the scratch cell) and `CHANGELOG.md`, per
   `plan.md` Step 18b's Do-list.
5. Locally, re-run `ANNOT=1 bash scripts/get_data.sh` so the annotation
   manifest's draft-availability counts reflect the repaired OCR (plan.md
   Step 18b Do-list point 4).
6. Commit, PR, merge, then Step 19 proceeds.

### If the session timed out mid-run

Nothing is lost, but read `FRESH_START` in the CONFIG cell first: attach this
run's own interrupted **Output** as an **Input** on the next push, and set
`FRESH_START = False` for that one push only — section 3 will then seed
`data/ocr/` from it and OCR resumes from where it stopped. Leaving
`FRESH_START = True` on a resume push would re-OCR everything from zero (safe,
just slow); leaving it `True` on the very first push is what you want, so the
default is correct as shipped.